In [1]:
# ==============================
# Carga de librerías y dataset
# ==============================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import copy
from torch.utils.data import DataLoader, TensorDataset

dataset = pd.read_pickle("../data/dataset_tfg.pkl").copy()
dataset = dataset.sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex)

alpha = 0.99
feature_cols = [c for c in dataset.columns if c != "target"]

In [2]:
def var_loss(y_true, y_pred, q=0.99, w=20.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, w, 1.0)
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)

In [3]:
class MLPVaR(nn.Module):
    def __init__(self, d_in, dropout=0.2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        nn.init.constant_(self.body[-1].bias, 0.02)

    def forward(self, x):
        return self.body(x)

In [4]:
def train_one_model(X_train, y_train, d_in, seed, w_loss,
                    alpha=0.99, max_epochs=200, lr=5e-5,
                    batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]

    model = MLPVaR(d_in=d_in)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
        batch_size=batch_size, shuffle=False
    )
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)

    best_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()

        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

In [5]:
from tqdm import tqdm

# ==============================
# Periodo de validación: 2010-2014
# Pesos a evaluar
# ==============================
w_list = [1, 5, 10, 15, 20, 25]

eval_start = pd.Timestamp("2010-01-01")
eval_end   = pd.Timestamp("2015-01-01")   # excluido: último día incluido = 2014-12-31
eval_dates = dataset.loc[eval_start:eval_end].index
# Filtramos para no incluir 2015
eval_dates = eval_dates[eval_dates < pd.Timestamp("2015-01-01")]

seed = 42

for w_val in w_list:
    print(f"\n{'='*50}")
    print(f"Entrenando MLP-VaR con w={w_val} | validación 2010-2014")
    print(f"{'='*50}")

    var_preds    = []
    real_targets = []
    dates        = []
    current_model = None
    muX, sdX = None, None

    for i, current_date in enumerate(
        tqdm(eval_dates, desc=f"w={w_val}", dynamic_ncols=True)
    ):
        train_end = current_date - pd.Timedelta(days=1)
        train_df  = dataset.loc[:train_end].tail(900)

        if len(train_df) < 250:
            if current_model is None:
                continue
        else:
            X_train = train_df[feature_cols].values.astype(np.float32)
            y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)

            muX = X_train.mean(axis=0, keepdims=True)
            sdX = X_train.std(axis=0, keepdims=True) + 1e-8
            X_train_norm = (X_train - muX) / sdX

            current_model = train_one_model(
                X_train_norm, y_train,
                d_in=X_train_norm.shape[1],
                seed=seed,
                w_loss=float(w_val)
            )

        if current_model is None or muX is None or sdX is None:
            continue

        test_df = dataset.loc[[current_date]]
        X_test  = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
        y_test  = test_df["target"].values.astype(np.float32).reshape(-1, 1)

        current_model.eval()
        with torch.no_grad():
            pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

        var_preds.append(pred)
        real_targets.append(y_test.reshape(-1)[0])
        dates.append(current_date)

    # Guardar resultados
    eval_df = pd.DataFrame(
        {"VaR_pred": var_preds, "loss_real": real_targets},
        index=pd.DatetimeIndex(dates)
    ).loc["2010":"2014"]

    eval_df["viol"] = (eval_df["loss_real"] > eval_df["VaR_pred"]).astype(int)
    eval_df["year"] = eval_df.index.year

    vr = eval_df["viol"].mean()
    print(f"  Obs: {len(eval_df)}  |  Violation Rate: {vr:.4f}  |  Violaciones: {eval_df['viol'].sum()}")

    out_path = f"../data/nn_val_predictions_{w_val}.csv"
    eval_df.to_csv(out_path)
    print(f"  Guardado: {out_path}")

print("\nTodos los pesos procesados.")


Entrenando MLP-VaR con w=1 | validación 2010-2014


w=1: 100%|██████████| 1256/1256 [19:55<00:00,  1.05it/s]


  Obs: 1256  |  Violation Rate: 0.0366  |  Violaciones: 46
  Guardado: ../data/nn_val_predictions_1.csv

Entrenando MLP-VaR con w=5 | validación 2010-2014


w=5:   3%|▎         | 33/1256 [01:05<40:30,  1.99s/it]


KeyboardInterrupt: 